# Bank Term Deposit Subscription Prediction
### Machine Learning Zoomcamp 2026 - Midterm Project

## 1. Problem Description
The dataset comes from direct marketing campaigns (phone calls) of a Portuguese banking institution.
The classification goal is to predict if the client will subscribe to a term deposit (`y` = `yes` / `no`).
Optimizing this prediction allows the bank to target high-propensity customers, minimizing marketing costs and unnecessary outreach while maximizing deposit acquisition.


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report


## 2. Data Preparation & Exploration

In [2]:
df = pd.read_csv("data/bank.csv", sep=";")
print("Dataset shape:", df.shape)
df.head()


Dataset shape: (4521, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,30,unemployed,married,primary,no,1787,no,no,cellular,19,oct,79,1,-1,0,unknown,no
1,33,services,married,secondary,no,4789,yes,yes,cellular,11,may,220,1,339,4,failure,no
2,35,management,single,tertiary,no,1350,yes,no,cellular,16,apr,185,1,330,1,failure,no
3,30,management,married,tertiary,no,1476,yes,yes,unknown,3,jun,199,4,-1,0,unknown,no
4,59,blue-collar,married,secondary,no,0,yes,no,unknown,5,may,226,1,-1,0,unknown,no


In [3]:
df["y"] = (df["y"] == "yes").astype(int)
print("Target distribution:")
print(df["y"].value_counts(normalize=True))
print("\nMissing values per column:")
print(df.isnull().sum())


Target distribution:
y
0    0.88476
1    0.11524
Name: proportion, dtype: float64

Missing values per column:
age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64


## 3. Dataset Splitting
Split into:
- 60% Train
- 20% Validation
- 20% Test


In [4]:
df_train_full, df_test = train_test_split(df, test_size=0.2, random_state=42, stratify=df["y"])
df_train, df_val = train_test_split(df_train_full, test_size=0.25, random_state=42, stratify=df_train_full["y"])

y_train = df_train["y"].values
y_val = df_val["y"].values
y_test = df_test["y"].values

del df_train["y"]
del df_val["y"]
del df_test["y"]

cat_cols = ["job", "marital", "education", "default", "housing", "loan", "contact", "month", "poutcome"]
num_cols = ["age", "balance", "day", "duration", "campaign", "pdays", "previous"]
features = cat_cols + num_cols
print(f"Train size: {len(df_train)}, Val size: {len(df_val)}, Test size: {len(df_test)}")


Train size: 2712, Val size: 904, Test size: 905


## 4. Feature Encoding with DictVectorizer

In [5]:
train_dicts = df_train[features].to_dict(orient="records")
val_dicts = df_val[features].to_dict(orient="records")
test_dicts = df_test[features].to_dict(orient="records")

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)
X_test = dv.transform(test_dicts)
print("Number of encoded features:", X_train.shape[1])


Number of encoded features: 51


## 5. Model Selection & Comparison
Evaluating 3 candidate models on the validation set using ROC-AUC:
1. Logistic Regression
2. Decision Tree Classifier
3. Random Forest Classifier


In [6]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred_proba = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, y_pred_proba)
    results[name] = auc
    print(f"{name} -> Validation ROC-AUC: {auc:.4f}")


Logistic Regression -> Validation ROC-AUC: 0.8925
Decision Tree -> Validation ROC-AUC: 0.7002
Random Forest -> Validation ROC-AUC: 0.9080


/home/congtran/.local/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## 6. Hyperparameter Tuning for Random Forest

In [7]:
tuning_scores = []
for depth in [6, 8, 10, 12]:
    for min_samples in [1, 3, 5]:
        rf = RandomForestClassifier(
            n_estimators=150,
            max_depth=depth,
            min_samples_leaf=min_samples,
            random_state=42,
            n_jobs=-1
        )
        rf.fit(X_train, y_train)
        y_val_proba = rf.predict_proba(X_val)[:, 1]
        auc = roc_auc_score(y_val, y_val_proba)
        tuning_scores.append({"max_depth": depth, "min_samples_leaf": min_samples, "roc_auc": round(auc, 4)})

tuning_df = pd.DataFrame(tuning_scores).sort_values(by="roc_auc", ascending=False)
tuning_df.head(5)


,max_depth,min_samples_leaf,roc_auc
9,12,1,0.9098
3,8,1,0.9094
6,10,1,0.9092
8,10,5,0.9052
11,12,5,0.9049


## 7. Final Model Evaluation on Test Set

In [8]:
best_rf = RandomForestClassifier(
    n_estimators=150,
    max_depth=10,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

# Train on full train (train + val)
df_train_full_copy = df_train_full.copy()
y_train_full = df_train_full_copy["y"].values
del df_train_full_copy["y"]

train_full_dicts = df_train_full_copy[features].to_dict(orient="records")
dv_final = DictVectorizer(sparse=False)
X_train_full = dv_final.fit_transform(train_full_dicts)
X_test_final = dv_final.transform(test_dicts)

best_rf.fit(X_train_full, y_train_full)
y_test_proba = best_rf.predict_proba(X_test_final)[:, 1]
final_auc = roc_auc_score(y_test, y_test_proba)
final_acc = accuracy_score(y_test, (y_test_proba >= 0.5).astype(int))

print(f"Final Test ROC-AUC: {final_auc:.4f}")
print(f"Final Test Accuracy: {final_acc:.4f}")


Final Test ROC-AUC: 0.9021
Final Test Accuracy: 0.8961


## 8. Feature Importance Analysis

In [9]:
importances = best_rf.feature_importances_
feature_names = dv_final.get_feature_names_out()
fi_df = pd.DataFrame({"feature": feature_names, "importance": importances}).sort_values(by="importance", ascending=False)
print("Top 10 Most Important Features:")
print(fi_df.head(10).to_string(index=False))


Top 10 Most Important Features:
         feature  importance
        duration    0.341554
poutcome=success    0.082928
             age    0.069795
         balance    0.056634
           pdays    0.051131
             day    0.050603
       month=oct    0.032694
        previous    0.030529
        campaign    0.021234
      housing=no    0.015205
